<a href="https://colab.research.google.com/github/DhikshaSubash/market-sentiment-analysis/blob/main/notebooks/02_data_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 - Install dependencies
!pip install yfinance newsapi-python kafka-python pyspark -q

print("✅ All libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.3/326.3 kB 5.7 MB/s eta 0:00:00
✅ All libraries installed!


In [2]:
# Cell 2 - Imports & Config
import yfinance as yf
from newsapi import NewsApiClient
import json
import time
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, FloatType, LongType

# ✅ Your config
NEWS_API_KEY = "c8945e883dae49868faf49adb8d7baa2"

# Stock symbols we'll track
STOCKS = ["AAPL", "GOOGL", "MSFT", "TSLA", "AMZN"]

# Start Spark
spark = SparkSession.builder \
    .appName("MarketSentimentIngestion") \
    .getOrCreate()

print("✅ Config ready!")
print(f"Tracking stocks: {STOCKS}")

✅ Config ready!
Tracking stocks: ['AAPL', 'GOOGL', 'MSFT', 'TSLA', 'AMZN']


In [3]:
# Cell 3 - Fetch live news
def fetch_news(stock_symbol):
    newsapi = NewsApiClient(api_key=NEWS_API_KEY)

    articles = newsapi.get_everything(
        q=stock_symbol,
        language='en',
        sort_by='publishedAt',
        page_size=10
    )

    news_list = []
    for article in articles['articles']:
        news_list.append({
            "symbol": stock_symbol,
            "headline": article['title'],
            "source": article['source']['name'],
            "published_at": article['publishedAt'],
            "ingested_at": datetime.now().isoformat()
        })

    return news_list

# Test with Apple
apple_news = fetch_news("AAPL")
print(f"✅ Fetched {len(apple_news)} news articles for AAPL")
print("\nSample headline:")
print(apple_news[0]['headline'])

✅ Fetched 10 news articles for AAPL

Sample headline:
openclaw-trading-assistant added to PyPI


In [4]:
# Cell 4 - Fetch live stock prices
def fetch_stock_price(symbol):
    ticker = yf.Ticker(symbol)
    hist = ticker.history(period="1d", interval="1m")

    price_list = []
    for timestamp, row in hist.iterrows():
        price_list.append({
            "symbol": symbol,
            "price": float(row['Close']),
            "volume": int(row['Volume']),
            "timestamp": str(timestamp)
        })

    return price_list

# Test with Apple
apple_prices = fetch_stock_price("AAPL")
print(f"✅ Fetched {len(apple_prices)} price points for AAPL")
print("\nLatest price data:")
print(apple_prices[-1])

✅ Fetched 390 price points for AAPL

Latest price data:
{'symbol': 'AAPL', 'price': 252.57000732421875, 'volume': 641365, 'timestamp': '2026-03-25 15:59:00-04:00'}


In [5]:
# Cell 5 - Simulate Kafka topics using Spark DataFrames

# Define schemas
news_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("headline", StringType(), True),
    StructField("source", StringType(), True),
    StructField("published_at", StringType(), True),
    StructField("ingested_at", StringType(), True)
])

price_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("price", FloatType(), True),
    StructField("volume", LongType(), True),
    StructField("timestamp", StringType(), True)
])

# Fetch all stocks
all_news = []
all_prices = []

for stock in STOCKS:
    print(f"Fetching data for {stock}...")
    all_news.extend(fetch_news(stock))
    all_prices.extend(fetch_stock_price(stock))
    time.sleep(1)  # be polite to APIs

# Create Spark DataFrames (simulating Kafka topics)
news_df = spark.createDataFrame(all_news, schema=news_schema)
price_df = spark.createDataFrame(all_prices, schema=price_schema)

print(f"\n✅ News Topic: {news_df.count()} records")
print(f"✅ Price Topic: {price_df.count()} records")

print("\n--- News Sample ---")
news_df.show(3, truncate=50)

print("\n--- Price Sample ---")
price_df.show(3)

Fetching data for AAPL...
Fetching data for GOOGL...
Fetching data for MSFT...
Fetching data for TSLA...
Fetching data for AMZN...

✅ News Topic: 50 records
✅ Price Topic: 1950 records

--- News Sample ---
+------+--------------------------------------------------+-------------------+--------------------+--------------------------+
|symbol|                                          headline|             source|        published_at|               ingested_at|
+------+--------------------------------------------------+-------------------+--------------------+--------------------------+
|  AAPL|          openclaw-trading-assistant added to PyPI|           Pypi.org|2026-03-25T03:18:37Z|2026-03-26T11:24:31.031994|
|  AAPL|Apple’s services flywheel spins at full speed, ...|   Macdailynews.com|2026-03-24T20:30:52Z|2026-03-26T11:24:31.032011|
|  AAPL|Morgan Stanley Is Doubling Down on Apple Stock ...|Yahoo Entertainment|2026-03-24T13:49:09Z|2026-03-26T11:24:31.032015|
+------+------------------

In [6]:
# Cell 6 - Save raw data to Drive
from google.colab import drive
drive.mount('/content/drive')

news_df.write.mode("overwrite").parquet("/content/drive/MyDrive/market_sentiment/raw/news")
price_df.write.mode("overwrite").parquet("/content/drive/MyDrive/market_sentiment/raw/prices")

print("✅ Raw data saved to Google Drive!")

Mounted at /content/drive
✅ Raw data saved to Google Drive!
